# Projeto de Implementação do Fuzzy C-Means

- Enzo Ambrósio (RA: )
- Giulia Monteiro Garrido (RA: 24010281)
- Thomaz Dacorso (RA: )

## Requisitos

**Requisitos do Projeto:**
1. Implementação e explicação do funcionamento do algoritmo (7pts)

2. Aplicar o algoritmo na base de dados Iris (completa, 4 atributos, 3 classes) e listar as pertinências de cada amostra usando 3 grupos (1pt)

3. Plot de resultados utilizando a base de dados Iris (apenas as amostras virgínica e versicolor) usando 2 grupos. Notem que é importante demonstrar a pertinência intermediária, o que será mais fácil em um problema de binário (2pts)

| **Parâmetros** | **Significado** |
| --- | --- |
| k | N° de grupos |
| m | Fator de difusão/fuzzificação |

| **Fórmulas/Funções** |
| --- |
| Criar pertinências aleatórias |
| Pertinência / w |
| Função objetivo |
| Atualizar centróides |

## Etapa 1

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import cdist

In [4]:
distancias = cdist(X, centroids, metric='euclidean')

NameError: name 'X' is not defined

In [14]:
def generate_random_relevence(X,k):
    
    '''
    Atribui a cada item uma relevância aleatória para cada grupo k.
    Garante que a soma de cada linha seja 1.
    '''

    relevences=[]    
    
    for i in range(len(X)):
        relevences.append([])
        aux=0
        sum=0

        for j in range(k):
            relevences[i].append(np.random.uniform(0,1-aux))
            aux+=relevences[i][j]
            sum+=relevences[i][j]

        if sum!=1:
            relevences[i][0]=1-sum+relevences[i][0]

    return np.array(relevences)
    


In [ ]:
def relevence (X,C,j,m):

    '''
    xi: um vetor
    C: matriz de centróides
    j: número do centróide
    m: parâmetro de fuzziness

    Calcula a pertinência/w de xi ao centróide C[j] e retorna 
    e retorna uma lista de pertinencias
    '''
    
    for i in X:

        exponent= 2/(m-1)
        divisor=(i-C[j])
        sum=0

        for l in C:
            denominator=(i-l)
            sum+= (divisor/denominator)**exponent
            
        return 1/sum

In [10]:
def centroid_update(X, C, U, m):

    n_clusters = U.shape[1]

    for j in range(n_clusters):
        sum_divisor = 0
        sum_denominator = 0

        for i in range(U.shape[0]):
            sum_divisor += (U[i][j]**m) * X[i]
            sum_denominator += U[i][j]**m

        C[j] = sum_divisor/sum_denominator
    
    return C

In [11]:
def objective_function(X,C,m):

    '''
    X: matriz de dados
    C: matriz de centróides
    m: parâmetro de fuzziness
    Calcula a função objetivo do FCM    
    '''

    n_samples = X.shape[0]
    n_clusters = C.shape[0]
    J = 0
    
    for i in range(n_samples):
        for j in range(n_clusters):
            u_ij = relevence(X[i], C, j, m)
            J += (u_ij ** m) * np.linalg.norm(X[i] - C[j]) ** 2

    return J

In [12]:
def fuzzy_cmeans(X, k, m, max_iter=100, conv = 0.001):

    n_samples = X.shape[0]

    random_indices = np.random.choice(n_samples, k, replace=False)
    C = X[random_indices].copy()

    U = generate_random_relevence(X, k)
    
    J_history = []

    for iteration in range(max_iter):
        C_old = C.copy()

        U_new = np.zeros_like(U)

        for i in range(n_samples):
            for j in range(k):
                U_new[i, j] = relevence(X[i], C, j, m)

        U = U_new

        C = centroid_update(X, C, U, m)
        J = objective_function(X, C, m)

        J_history.append(J)
        convergence = np.linalg.norm(C - C_old)

        if convergence < conv:
            print(f"Convergiu na iteração {iteration}")
            break
    
    return U, C, J_history

## Etapa 2

In [15]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data

U, C, J = fuzzy_cmeans(X, k=3, m=2)

print("Pertinências:", U)
print("Centróides:", C)

ValueError: setting an array element with a sequence.